Inports

In [1]:
pip install pyodbc

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install -U imbalanced-learn

Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install -U scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [4]:
import pandas as pd
#Ig
from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest
# Chi square
from sklearn.feature_selection import chi2
import numpy as np
# SU
import math
from scipy.stats import entropy
from sklearn import metrics

In [5]:
class ModelResult:
    def __init__(self, level_measure,project,b,rank,model_selection,intance_reduction,model,precision,recall,f,accuracy,count,pack):
        self.level_measure = level_measure
        self.project = project
        self.b = b
        self.rank = rank
        self.model_selection = model_selection
        self.intance_reduction = intance_reduction
        self.model = model
        self.precision = precision
        self.recall = recall
        self.f = f
        self.accuracy = accuracy
        self.count = count
        self.pack = pack
class SelectMeasure:
    def __init__(self,level_measure,project,b,rank,pack,cs_su,ig_su,rf_su,cs_cos,ig_cos,rf_cos):
        self.level_measure = level_measure
        self.project = project
        self.b = b
        self.rank = rank
        self.pack = pack
        self.cs_su = cs_su
        self.ig_su = ig_su
        self.rf_su = rf_su
        self.cs_cos = cs_cos
        self.ig_cos = ig_cos
        self.rf_cos = rf_cos

In [6]:
import os
def FileList(directory,list=[]):
  for filename in os.listdir(directory):
      f = os.path.join(directory, filename)
      if os.path.isdir(f):
        FileList(f,list)
      # checking if it is a file
      if os.path.isfile(f) and os.path.splitext(f)[-1].lower() == '.csv':
        list.append(f)
  return np.array(list)

In [7]:
def GetMetrics(ytest, ypred,name):
    return pd.DataFrame([
        metrics.precision_score(ytest, ypred, average='weighted'),
        metrics.recall_score(ytest, ypred, average='weighted'),
        metrics.f1_score(ytest, ypred, average='weighted'),
        metrics.accuracy_score(ytest, ypred)
    ],
                    index=["pr","re","f","Ac"],
                    columns=[name])

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import confusion_matrix

def NaiveBayes(X,y,name):
  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=69)
  model = MultinomialNB().fit(X_train, y_train)
  y_pred = model.predict(X_test)
  return GetMetrics(y_test, y_pred,name)

In [9]:
# from sklearn.neighbors import KNeighborsClassifier
# from sklearn.datasets import load_iris
# from sklearn.datasets import make_classification
# from sklearn.model_selection import train_test_split
# from sklearn.metrics import confusion_matrix

# from sklearn.metrics import classification_report# Iris dataset example 

 

# iris = load_iris()
# x, y = iris.data, iris.target

# xtrain, xtest, ytrain, ytest=train_test_split(x, y, test_size=0.10)


# knc = KNeighborsClassifier(n_neighbors = 3)
# print(knc)

# knc.fit(xtrain, ytrain)
# score = knc.predic(xtrain, ytrain)
# print("Score: ", score)
# cr = classification_report(ytest, ypred)
# print(cr)

# print("recall :" ,metrics.recall_score(ytest, ypred, average='weighted'))
# print("Precision :" ,metrics.precision_score(ytest, ypred, average='weighted'))
# print("f_score :" ,metrics.f1_score(ytest, ypred, average='weighted'))
# print("Acurrency:" ,metrics.accuracy_score(ytest, ypred))
# ypred = knc.predict(xtest)



In [10]:
from sklearn.neighbors import KNeighborsClassifier
def KNN(X,y,name):
  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=69)
  knn = KNeighborsClassifier(n_neighbors=5)
  knn.fit(X_train, y_train)
  y_pred = knn.predict(X_test)
  return GetMetrics(y_test, y_pred,name)

In [11]:
from sklearn.tree import DecisionTreeClassifier # Import Decision Tree Classifier
def DecisionTree(X,y,name):
  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=69)
  clf = DecisionTreeClassifier()
  clf = clf.fit(X_train,y_train)
  y_pred = clf.predict(X_test)
  return GetMetrics(y_test, y_pred,name)

In [12]:
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler

def ExecuteModel(model,IntanceReduction,X,y,column):
  try:
    if IntanceReduction == "RUS":
       rus = RandomUnderSampler(random_state=0)
       X, y = rus.fit_resample(X, y)
    elif IntanceReduction == "ROS":
       ros = RandomOverSampler(random_state=0)
       X, y = ros.fit_resample(X, y)

    # Algoritmo a ejecutar
    if model == "NB":
      return NaiveBayes(X,y,column)
    elif model == "KNN":
      return KNN(X,y,column)
    elif model == "DT":
      return DecisionTree(X,y,column)
  except Exception as e:
     print(str(e))
     return pd.DataFrame([-1,-1,-1,-1],
                    index=["pr","re","f","Ac"],
                    columns=[column])


In [13]:
import pyodbc 
def insertDb(data):
    conn = pyodbc.connect('Driver={SQL Server};'
                      'Server=JORDANYS;'
                      'Database=magister;'
                      'Trusted_Connection=yes;')
    cursor = conn.cursor()
  
    dataInsert = "'{level_measure}','{project}','{b}','{rank}','{model_selection}','{intance_reduction}','{model}','{precision}','{recall}','{f}','{accuracy}','{count}','{pack}'".format(
        level_measure = data.level_measure ,
        project = data.project,
        b = data.b,
        rank = data.rank,
        model_selection = data.model_selection,
        intance_reduction = data.intance_reduction,
        model = data.model,
        precision = data.precision,
        recall = data.recall,
        f = data.f,
        accuracy = data.accuracy,
        count = data.count,
        pack = data.pack
    )
 
    cursor.execute('''
                INSERT INTO model_results_pack ([level_measure],[project],[b],[rank],[model_selection],[intance_reduction],[model],[precision],[recall],[f],[accuracy],[count],[pack])
                VALUES
                ('''+ dataInsert+''')

                ''')

    conn.commit()

In [14]:
def SelectMetricsDb(data):
    conn = pyodbc.connect('Driver={SQL Server};'
                      'Server=JORDANYS;'
                      'Database=magister;'
                      'Trusted_Connection=yes;')
    cursor = conn.cursor()
   
    query = "SELECT DISTINCT metric FROM [magister].[dbo].[select_measure_pack] where pack = {pack} and b = {b} and rank = {rank} and project = '{project}' and level_measure = '{level_measure}' and cs_su like '%{cs_su}%' and ig_su like '%{ig_su}%' and rf_su like '%{rf_su}%' and cs_cos like '%{cs_cos}%' and ig_cos like '%{ig_cos}%' and rf_cos like '%{rf_cos}%'".format(
        level_measure = data.level_measure,
        project = data.project,
        b = data.b,
        rank = data.rank,
        pack = data.pack,
        cs_su = data.cs_su,
        ig_su = data.ig_su,
        rf_su = data.rf_su,
        cs_cos = data.cs_cos,
        ig_cos = data.ig_cos,
        rf_cos = data.rf_cos
      
    )
    cursor.execute(query)
    results = cursor.fetchall()
    return [i[0] for i in results] 

In [15]:
def SelectMetricsDbCSV(data,dataCSV):
        return dataCSV.loc[(dataCSV['b'] == data.b) 
                       & (dataCSV['pack'] == data.pack)
                       & (dataCSV['rank'] == data.rank)
                       & (dataCSV['project'] == data.project)
                       & (dataCSV['level_measure'] == data.level_measure)
                       & ((dataCSV['cs_su'].astype(int) == data.cs_su)
                       | (dataCSV['ig_su'].astype(int) == data.ig_su)
                       | (dataCSV['rf_su'].astype(int) == data.rf_su)
                       | (dataCSV['cs_cos'].astype(int) == data.cs_cos)
                       | (dataCSV['ig_cos'].astype(int) == data.ig_cos)
                       | (dataCSV['rf_cos'].astype(int) == data.rf_cos))
                      ]['metric'].unique()

In [16]:
def SelectPackDb(data):
    conn = pyodbc.connect('Driver={SQL Server};'
                      'Server=JORDANYS;'
                      'Database=magister;'
                      'Trusted_Connection=yes;')
    cursor = conn.cursor()
   
    query = "SELECT DISTINCT pack FROM [magister].[dbo].[select_measure_pack] where b = {b} and rank = {rank} and project = '{project}' and level_measure = '{level_measure}' and cs_su like '%{cs_su}%' and ig_su like '%{ig_su}%' and rf_su like '%{rf_su}%' and cs_cos like '%{cs_cos}%' and ig_cos like '%{ig_cos}%' and rf_cos like '%{rf_cos}%'".format(
        level_measure = data.level_measure,
        project = data.project,
        b = data.b,
        rank = data.rank,
        pack = data.pack,
        cs_su = data.cs_su,
        ig_su = data.ig_su,
        rf_su = data.rf_su,
        cs_cos = data.cs_cos,
        ig_cos = data.ig_cos,
        rf_cos = data.rf_cos
    )
    cursor.execute(query)
    results = cursor.fetchall()
    return [i[0] for i in results] 

In [17]:
def SelectPackDbCSV(data,dataCSV):
    print(data)
    return dataCSV.loc[(dataCSV['b'] == data.b) 
                       & (dataCSV['rank'] == data.rank)
                       & (dataCSV['project'] == data.project)
                       & (dataCSV['level_measure'] == data.level_measure)
                       & ((dataCSV['cs_su'].astype(int) == data.cs_su)
                          | (dataCSV['ig_su'].astype(int) == data.ig_su)
                       | (dataCSV['rf_su'].astype(int) == data.rf_su)
                       | (dataCSV['cs_cos'].astype(int) == data.cs_cos)
                       | (dataCSV['ig_cos'].astype(int) == data.ig_cos)
                       | (dataCSV['rf_cos'].astype(int) == data.rf_cos))
                      ]['pack'].unique()

In [18]:
directory = 'C:/Users/danyr/Downloads/Python/full/'
listFile = FileList(directory)
projects =np.unique([os.path.split(os.path.dirname(i))[1] for i in listFile])
print(projects)

['Android-Universal-Image-Loader' 'BroadleafCommerce' 'MapDB' 'all'
 'antlr4' 'ceylon-ide-eclipse' 'elasticsearch' 'hazelcast' 'junit' 'mcMMO'
 'mct' 'neo4j' 'netty' 'orientdb' 'oryx' 'titan']


In [19]:
def ModelsByCondition(project,dataCSV):
    ranks = [0.5]
    bs = [0.9]
    models = ['NB','KNN','DT']
    modelsSelections = ['ALL','CS_SU','IG_SU','RF_SU','CS_COS','IG_COS','RF_COS']
    IntanceReductions = ['RUS','ROS','NONE']
    for rank in ranks:
          for b in bs:
                paths = [i for i in listFile if os.path.split(os.path.dirname(i))[1] == project] 
                for path in paths:
                    data=pd.read_csv(path)
                    try:
                      data=data.drop(columns=['Project','Hash', 'LongName','Parent'],axis=1, errors='ignore')
                      X =data.drop(labels=['Number of Bugs'], axis=1).astype(float).abs()
                      y =  [1 if i > 0 else 0 for i in data['Number of Bugs']]
                    except Exception as e:
                      print("Error in " + file_location + str(e))
                      continue
                    for model in models:
                        for modelsSelection in modelsSelections:
                            for reduction in IntanceReductions:
                              level = os.path.split(os.path.basename(path))[1].split('.')[0] 
                              measure = SelectMeasure(level,project,b,rank,'','','','','','','')
                              if modelsSelection == 'CS_SU':
                                    measure.cs_su = 1
                              elif modelsSelection == 'IG_SU':
                                    measure.ig_su = 1
                              elif modelsSelection == 'RF_SU':
                                    measure.rf_su = 1
                              elif modelsSelection == 'CS_COS':
                                    measure.cs_cos = 1
                              elif modelsSelection == 'IG_COS':
                                    measure.ig_cos = 1
                              elif modelsSelection == 'RF_COS':
                                    measure.rf_cos = 1
                              packs = SelectPackDbCSV(measure,dataCSV)
                              print(packs)
                              for pack in packs:
                                  measure.pack = pack
                                  metricsList = SelectMetricsDbCSV(measure,dataCSV) 
                                  result = ExecuteModel(model,reduction,X[metricsList],y,modelsSelection)
                                  insertDb(ModelResult(level,project,b,rank,modelsSelection,reduction,model,result[modelsSelection]['pr'],result[modelsSelection]['re'],result[modelsSelection]['f'],result[modelsSelection]['Ac'],len(metricsList),pack))
#                               print(result[modelsSelection]['Ac'],modelsSelection,model,project,len(metricsList))
 

In [20]:
dataCSVFILE=pd.read_csv("metricData.csv").dropna()

In [21]:
# # print(dataCSVFILE)
# for project in projects:
#     ModelsByCondition(project,dataCSVFILE)

In [ ]:
from joblib import Parallel, delayed
Parallel(n_jobs=10)(delayed(ModelsByCondition)(project,dataCSVFILE) for project in projects)